# Disease-Agnostic Spatial-MicroCKG Agent
## TBI Proof-of-Concept — Panel Demo

**Objective:** Demonstrate a disease-agnostic spatial multi-omics pipeline that
extracts highly stable neuroinflammation signatures from spatial transcriptomics
data, structures them into a Micro-Clinical Knowledge Graph, and provides
evidence-traced LLM-driven analysis.

### Architecture

```
Spatial Data (.h5ad)
       │
       ▼
QC & Normalization (scanpy)
       │
       ▼
2,000 Highly Variable Genes
       │
       ▼
Stabl Feature Selection
(L1 LogReg, 500 bootstraps, FDP+ threshold)
       │
       ▼
N Stable Biomarker Genes (cached)
       │
       ▼
BioCypher Micro-CKG
(Gene ↔ CellType ↔ AnatomicalEntity)
       │
       ▼
LangChain QA Agent (Gemini)
with Evidence Traceability
```

This pipeline is designed as a **Disease-Agnostic Neuroinflammation Infrastructure**,
readily adaptable to internal drug discovery datasets across therapeutic areas.

## 1. Environment Setup

In [ ]:
import sys
from pathlib import Path

# Ensure src/ is importable
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

# Core imports
from src.data_ingestion import get_dataset
from src.spatial_pipeline import (
    load_adata,
    run_qc,
    normalize,
    run_stabl_cached,
    plot_spatial_markers,
)
from src.biocypher_adapter import build_micro_ckg
from src.llm_agent import create_qa_agent, query_graph

DATA_DIR = PROJECT_ROOT / "data" / "raw"
CACHE_DIR = PROJECT_ROOT / "cache"
ASSETS_DIR = PROJECT_ROOT / "assets"

print("Environment ready.")

## 2. Data Ingestion

Download the 10x Visium adult mouse brain spatial transcriptomics dataset.
This dataset includes spatial coordinates and an H&E tissue image,
enabling anatomical alignment of gene expression patterns.

In [ ]:
h5ad_path = get_dataset(DATA_DIR, source="squidpy")
adata = load_adata(h5ad_path)
print(f"\nDataset: {adata.shape[0]} spots × {adata.shape[1]} genes")

## 3. Quality Control & Preprocessing

Apply standard QC filters: remove low-quality spots (< 200 genes detected),
rarely expressed genes (< 3 cells), and spots with elevated mitochondrial
gene content (> 5%). Normalize to 10,000 counts per cell and log-transform.

In [ ]:
adata = run_qc(adata)
adata = normalize(adata)
print(f"\nPost-QC: {adata.shape[0]} spots × {adata.shape[1]} genes")

## 4. Stabl Feature Selection (Cached)

**Protocol:**
1. Isolate 2,000 Highly Variable Genes (HVGs) from the normalized matrix.
2. Apply the Stabl algorithm (L1-penalised logistic regression with stability
   selection across 500 bootstrap iterations) to objectively converge this
   2,000-feature pool down to the most stable biomarker genes.
3. The FDP+ (False Discovery Proportion) threshold is determined automatically
   by Stabl — no manual override is applied.

Results are cached to disk. On subsequent runs, cached results load in under
one second.

In [ ]:
stabl_result = run_stabl_cached(
    adata,
    cache_dir=CACHE_DIR,
    dataset_name="squidpy",
    n_hvgs=2000,
    label_method="cluster",
    n_bootstraps=500,
)

print(f"\nStabl results:")
print(f"  Features selected: {stabl_result['n_selected']}")
print(f"  FDP+ threshold: {stabl_result['threshold']:.4f}")
print(f"  Minimum FDP+: {stabl_result['fdr']:.4f}")

In [ ]:
import pandas as pd

# Display selected features sorted by stability score
df_features = pd.DataFrame({
    "Gene": stabl_result["selected_genes"],
    "Stability Score": [
        stabl_result["stability_scores"][g]
        for g in stabl_result["selected_genes"]
    ],
}).sort_values("Stability Score", ascending=False).reset_index(drop=True)

print(f"Top Stabl-certified biomarker genes ({len(df_features)} total):")
df_features.head(20)

## 5. Spatial Visualization — Anatomical Proof

Overlay the expression of the top Stabl-certified marker genes directly
onto the H&E tissue image. This provides spatial anatomical alignment,
confirming that selected biomarkers localise to biologically coherent
tissue regions.

In [ ]:
# Sort markers by stability score (descending)
top_markers = df_features["Gene"].tolist()

saved_plots = plot_spatial_markers(
    adata,
    markers=top_markers,
    save_dir=ASSETS_DIR,
    n_top=5,
)

print(f"\n{len(saved_plots)} spatial plots saved to {ASSETS_DIR}")

In [ ]:
# Display spatial plots inline
from IPython.display import Image, display

for plot_path in saved_plots:
    display(Image(filename=str(plot_path), width=500))

## 6. BioCypher Micro-CKG Construction

Construct a Micro-Clinical Knowledge Graph using the Biolink model ontology.

- **Nodes:** Gene (with stability scores), CellType (Leiden clusters),
  AnatomicalEntity (brain regions)
- **Edges:** Quantifiable associations driven by Stabl stability scores,
  mean cluster expression, and spatial correlation metrics.

In [ ]:
schema_path = PROJECT_ROOT / "config" / "schema_config.yaml"

graph = build_micro_ckg(
    stabl_result=stabl_result,
    adata=adata,
    schema_path=schema_path,
)

print(f"\nMicro-CKG constructed:")
print(f"  Total nodes: {graph.number_of_nodes()}")
print(f"  Total edges: {graph.number_of_edges()}")

## 7. LangChain Graph QA Agent

Query the Micro-CKG through an LLM agent with strict evidence traceability.
The agent uses Google Gemini and is required to cite exact source nodes,
edges, and stability scores for every claim.

**Anti-Hallucination Rule:** If no evidence exists in the graph, the agent
responds: *"No evidence found in the Micro-CKG for this query."*

In [ ]:
agent = create_qa_agent(graph, provider="google")
print("QA agent initialised (Google Gemini).")

In [ ]:
# Query 1: Top neuroinflammation markers
answer1 = query_graph(
    agent,
    "What are the top 5 genes with the highest stability scores, "
    "and which cell types and brain regions are they associated with?"
)
print(answer1)

In [ ]:
# Query 2: Cell type associations
answer2 = query_graph(
    agent,
    "Which genes are most associated with cortical cell types? "
    "List the evidence with stability and expression scores."
)
print(answer2)

In [ ]:
# Query 3: Regional specificity
answer3 = query_graph(
    agent,
    "Are there genes that show region-specific expression patterns? "
    "Identify genes with high spatial correlation in specific anatomical regions."
)
print(answer3)

## 8. Summary

This notebook demonstrated a complete **Disease-Agnostic Neuroinflammation
Infrastructure** pipeline:

1. **Automated Data Ingestion:** Programmatic download of spatial
   transcriptomics data in `.h5ad` format.
2. **Objective Feature Convergence:** 2,000 HVGs reduced to N stable
   biomarkers via Stabl's FDP+-controlled stability selection.
3. **Spatial Anatomical Proof:** H&E overlay plots confirming biologically
   coherent marker localisation.
4. **Ontology-Mapped Micro-CKG:** Gene, CellType, and AnatomicalEntity
   nodes with quantifiable association edges.
5. **Evidence-Traced LLM Queries:** Every answer backed by explicit graph
   evidence with numerical scores.

The infrastructure is readily extensible to internal drug discovery datasets
across neuroinflammation, neurodegeneration, and other therapeutic areas.
Replacing the input `.h5ad` file and adjusting label assignment enables
immediate application to new disease contexts.